# Catfish Water Quality Prediction Model
This notebook loads water quality data, preprocesses it, trains a machine learning model, and predicts water quality status for catfish farming.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Set plot style
sns.set(style="whitegrid")

## Problem Definition & ML Strategy

### The Problem
The dataset (`WQD.xlsx`) contains various chemical and biological water quality measurements, such as temperature, dissolved oxygen (DO), pH, and turbidity. The final column, "Water Quality," is our target variable.

Based on the snippets, the "Water Quality" column contains discrete integer values (e.g., 0, 2), indicating this is a **classification problem**. The goal is to build a model that can predict the water quality class based on the other measurements.

### How ML Can Help
A machine learning classification model can learn the complex, non-linear relationships between the different measurements and the resulting water quality.

* **Prediction:** Once trained, the model can instantly classify new water samples, providing a real-time monitoring tool.
* **Insight:** The model can help identify which factors (e.g., 'BOD', 'Turbidity') are the most important predictors of poor water quality.
* **Automation:** This automates a potentially slow and complex manual assessment process, allowing for faster responses to pollution events.

In [6]:
# Load the dataset
file_name = 'WQD.xlsx'
df = pd.read_excel(file_name)

# --- Data Cleaning (from snippet inspection) ---
# The 'pH`' column in the snippet has a trailing backtick. Let's fix it.
if 'pH`' in df.columns:
    df = df.rename(columns={'pH`': 'pH'})

# Display first few rows and data types
print(df.head())
print("\n" + "="*50 + "\n")
print(df.info())
print(df.describe())

        Temp  Turbidity (cm)   DO(mg/L)  BOD (mg/L)        CO2         pH  \
0  67.448725       10.127148   0.208153    7.473607  10.181084   4.751657   
1  64.626666       94.015595  11.434463   10.859998  14.860521   3.085154   
2  65.121842       90.653462  12.430865   12.809970  12.319980   9.648515   
3   1.640334        0.066344  10.963529    8.508023  12.955209   4.819988   
4  64.863434        2.119173   1.361736   13.335372  13.603197  10.244034   

   Alkalinity (mg L-1 )  Hardness (mg L-1 )  Calcium (mg L-1 )  \
0            218.364855          300.125080         337.178226   
1            273.939692            8.426776         363.660740   
2            220.812730           11.726274         309.370934   
3            266.571628            6.627655           8.180468   
4            252.108000          339.891514         253.996871   

   Ammonia (mg L-1 )  Nitrite (mg L-1 )  Phosphorus (mg L-1 )  H2S (mg L-1 )  \
0           0.286054           4.355310              0.00598

In [7]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Simple imputation: fill missing values with the median of the column
# (Median is more robust to outliers than mean)
for col in df.columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled NaNs in '{col}' with median: {median_val}")

# Define Features (X) and Target (y)
target = 'Water Quality'
X = df.drop(target, axis=1)
y = df[target]

# Check target variable distribution
print("\nTarget variable distribution:")
print(y.value_counts())

Missing values per column:
Temp                    0
Turbidity (cm)          0
DO(mg/L)                0
BOD (mg/L)              0
CO2                     0
pH                      0
Alkalinity (mg L-1 )    0
Hardness (mg L-1 )      0
Calcium (mg L-1 )       0
Ammonia (mg L-1 )       0
Nitrite (mg L-1 )       0
Phosphorus (mg L-1 )    0
H2S (mg L-1 )           0
Plankton (No. L-1)      0
Water Quality           0
dtype: int64

Target variable distribution:
Water Quality
2    1500
1    1400
0    1400
Name: count, dtype: int64
